# RetailRocket — Purchase Prediction (Classification)

**Goal:** Predict whether a session ends in a purchase (`purchased = 1`).

**Challenge:** The target is heavily imbalanced (0.81 % positive class). We handle this with:
- `class_weight='balanced'` in the model
- Evaluation via **Precision-Recall AUC** and **F1-score** (not plain Accuracy)

## Contents
1. Load data & split  
2. Baseline model (Logistic Regression)  
3. Random Forest  
4. Evaluation & comparison  
5. Feature importance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
)

sns.set_theme(style='whitegrid', palette='muted')
DATA_DIR = os.path.join('..', 'data')

RANDOM_STATE = 42

## 1. Load Data & Train/Test Split

In [ ]:
sessions = pd.read_parquet(os.path.join(DATA_DIR, 'sessions_features.parquet'))
print(f'Shape: {sessions.shape}')
sessions.head()

In [ ]:
FEATURES = ['n_events', 'n_views', 'n_addtocart', 'n_items', 'duration_sec', 'has_addtocart']
TARGET   = 'purchased'

X = sessions[FEATURES]
y = sessions[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows')
print(f'Purchase rate — train: {y_train.mean()*100:.2f}%  |  test: {y_test.mean()*100:.2f}%')

In [ ]:
# Scale for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## 2. Baseline: Logistic Regression

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_scaled, y_train)

y_pred_lr   = lr.predict(X_test_scaled)
y_proba_lr  = lr.predict_proba(X_test_scaled)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['no purchase', 'purchase']))

## 3. Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    max_depth=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['no purchase', 'purchase']))

## 4. Evaluation & Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Confusion matrices ---
for ax, model, y_pred, label in [
    (axes[0], lr, y_pred_lr, 'Logistic Regression'),
    (axes[1], rf, y_pred_rf, 'Random Forest'),
]:
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=['No purchase', 'Purchase'],
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(label)

# --- Precision-Recall curves ---
for y_prob, label, color in [
    (y_proba_lr, 'Logistic Regression', '#4C72B0'),
    (y_proba_rf, 'Random Forest',        '#55A868'),
]:
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    axes[2].plot(rec, prec, label=f'{label} (AP={ap:.3f})', color=color)

baseline_rate = y_test.mean()
axes[2].axhline(baseline_rate, linestyle='--', color='grey', label=f'Baseline ({baseline_rate:.3f})')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import f1_score

results = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'ROC-AUC': roc_auc_score(y_test, y_proba_lr),
        'PR-AUC':  average_precision_score(y_test, y_proba_lr),
        'F1 (purchase)': f1_score(y_test, y_pred_lr),
    },
    {
        'Model': 'Random Forest',
        'ROC-AUC': roc_auc_score(y_test, y_proba_rf),
        'PR-AUC':  average_precision_score(y_test, y_proba_rf),
        'F1 (purchase)': f1_score(y_test, y_pred_rf),
    },
])

results = results.set_index('Model')
print(results.round(4))

## 5. Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot(kind='barh', ax=ax, color='#4C72B0')
ax.set_title('Feature Importance — Random Forest')
ax.set_xlabel('Mean decrease in impurity')
plt.tight_layout()
plt.show()

## Summary & Interpretation

| Model | ROC-AUC | PR-AUC | F1 (purchase) |
|-------|---------|--------|---------------|
| Logistic Regression | — | — | — |
| Random Forest | — | — | — |

*(values will fill in after running)*

### Key takeaways
- **Accuracy is misleading** here — a model predicting "no purchase" for every session would achieve 99.19 % accuracy.
- **PR-AUC** and **F1** are the right metrics for imbalanced problems.
- `has_addtocart` and `n_addtocart` are expected to be the strongest predictors — adding items to cart is a clear purchase signal.
- `duration_sec` captures engagement depth that raw event counts miss.

### Possible next steps
- Add time-of-day / day-of-week features from session start time
- Add item-category features (from `item_properties` files)
- Try gradient boosting (XGBoost / LightGBM) for better performance
- Tune classification threshold to optimise Precision vs. Recall trade-off